# CO source and DSB features

## Setup

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib.patches as patches
from matplotlib import colormaps
from matplotlib.patches import PathPatch
from matplotlib.path import Path as MplPath
from matplotlib.lines import Line2D
import matplotlib.cm
from pathlib import Path
import tqdm
import sys
import seaborn as sns
import scipy.stats
import msprime
import os
import joblib
import polars as pl
import glob
import logging

os.environ["PATH"] += ":" + os.path.join(sys.prefix, "bin")
os.environ["PATH"] += ":" + "/software/treeoflife/shpc/0.1.26/wrapper/quay.io/biocontainers/bedtools/2.31.1--hf5e1c6e_1/bin"
import pybedtools

pd.set_option('display.max_rows', 1000)
pl.Config.set_tbl_rows(-1)
pl.Config.set_fmt_str_lengths(50)
plt.rcParams["pdf.use14corefonts"] = True
plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["font.sans-serif"] = ["DejaVu Sans"]
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)

In [3]:
repo = Path(os.getcwd())
if repo.name == "notebooks":
    repo = repo.parent
elif not (repo / "configs").exists():
    repo = Path("/nfs/users/nfs_r/rs42/rs42/git/recombination")

sys.path.append(str(repo))
from src.IDs import *
from src import liftover

aut_chrom_names = [f"chr{i}" for i in list(range(1, 23))]

grch37_chromosome_sizes_in_bp = {
    'chr1': 249250621,
    'chr2': 243199373,
    'chr3': 198022430,
    'chr4': 191154276,
    'chr5': 180915260,
    'chr6': 171115067,
    'chr7': 159138663,
    'chr8': 146364022,
    'chr9': 141213431,
    'chr10': 135534747,
    'chr11': 135006516,
    'chr12': 133851895,
    'chr13': 115169878,
    'chr14': 107349540,
    'chr15': 102531392,
    'chr16': 90354753,
    'chr17': 81195210,
    'chr18': 78077248,
    'chr19': 59128983,
    'chr20': 63025520,
    'chr21': 48129895,
    'chr22': 51304566,
}

grch38_chromosome_sizes_in_bp = {
    'chr1': 248_956_422,
    'chr2': 242_193_529,
    'chr3': 198_295_559,
    'chr4': 190_214_555,
    'chr5': 181_538_259,
    'chr6': 170_805_979,
    'chr7': 159_345_973,
    'chr8': 145_138_636,
    'chr9': 138_394_717,
    'chr10': 133_797_422,
    'chr11': 135_086_622,
    'chr12': 133_275_309,
    'chr13': 114_364_328,
    'chr14': 107_043_718,
    'chr15': 101_991_189,
    'chr16': 90_338_345,
    'chr17': 83_257_441,
    'chr18': 80_373_285,
    'chr19': 58_617_616,
    'chr20': 64_444_167,
    'chr21': 46_709_983,
    'chr22': 50_818_468,
    'chrX': 156_040_895,
    'chrY': 57_227_415,
}

rate_maps = {}
for chrom in aut_chrom_names:
    rate_maps[chrom] = msprime.RateMap.read_hapmap(
        open(f"/lustre/scratch122/tol/projects/sperm/data/references/04.genetic_maps/01.Bherer_etal_SexualDimorphismRecombination/Refined_EUR_genetic_map_b37/male_{chrom}.txt"),
        sequence_length=grch37_chromosome_sizes_in_bp[chrom],
    )

figure_dir = repo / "figures"
figure_dir.mkdir(parents=True, exist_ok=True)

CO_color = globals().get("CO_color", "#4C78A8")
NCO_color = globals().get("NCO_color", "#54A24B")
background_color = globals().get("background_color", "#777777")

events_xlsx = Path("/lustre/scratch122/tol/projects/sperm/results/recombination_events_sperm_20250417.xlsx")
events_parquet = Path("/lustre/scratch122/tol/projects/sperm/results/recombination_events_sperm_20250325.parquet")
accepted_events_df = pl.read_excel(events_xlsx)
events_df = pl.read_parquet(events_parquet)

Could not determine dtype for column 8, falling back to string
Could not determine dtype for column 9, falling back to string
Could not determine dtype for column 10, falling back to string
Could not determine dtype for column 16, falling back to string
Could not determine dtype for column 17, falling back to string


In [4]:
rahbari_df = pl.read_csv(repo / "configs/Rahbari.tsv", separator="\t")

sudmant_df = (
    pl.read_csv(repo / "configs/Sudmant.tsv", separator="\t")
    .with_columns(
        pl.col("sample_set").cast(pl.String),
        pl.col("sample_id").cast(pl.String),
        pl.col("flow_cell").cast(pl.String),
    )
)

CEPH_df = pl.read_csv(repo / "configs/CEPH.tsv", separator="\t")
ceph_df = CEPH_df

ceph_sample_ids_no_gps = sorted([
    x for x in CEPH_df["sample_id"].unique().to_list()
    if x not in ["NA12889", "NA12890", "NA12891", "NA12892"]
])

sample_ids = rahbari_sample_ids + sudmant_sample_ids

## Events

In [5]:
%%time
reads_filenames = (
    [
        (
            f"/lustre/scratch122/tol/projects/sperm/results/Rahbari_20250212/read_analysis/{sample_set}/{sample_id}/reads/{chrom}/all_reads_structure_annotated.parquet"
        )
        for sample_id, sample_set in tqdm.tqdm(rahbari_df.select("sample_id", "sample_set").unique().iter_rows())
        for chrom in aut_chrom_names
    ] +
    [
        (
            f"/lustre/scratch122/tol/projects/sperm/results/Sudmant_20241121/read_analysis/{sample_set}/{sample_id}/reads/{chrom}/all_reads_structure_annotated.parquet"
        )
        for sample_id, sample_set in tqdm.tqdm(sudmant_df.select("sample_id", "sample_set").unique().iter_rows())
        for chrom in aut_chrom_names
    ]
)

9it [00:00, 39486.13it/s]
6it [00:00, 37843.34it/s]

CPU times: user 7.46 ms, sys: 16.6 ms, total: 24.1 ms
Wall time: 9.91 ms


## CO source subset

In [6]:
%%time
def read_CO_source(filename):
    return (
        pl.scan_parquet(filename)
        .select(
            'read_name',
            'read_length',
            'chrom',
            'sample_id',
            'high_quality_snp_positions',
            'grch37_reference_start',
            'grch38_reference_start',
            'grch37_reference_end',
            'grch38_reference_end',
            'CO_active_interval_crossover_prob',
            'full_read_crossover_prob',
            'AA_motif_center_pos',
            'is_high_quality_read',
            'high_quality_classification_class',
            'snp_positions_on_read',
            'idx_transitions',
            'high_quality_classification_in_detectable_class',
            'is_contamination',
        )
        .filter(pl.col("high_quality_classification_in_detectable_class") == "CO")
        .filter(pl.col("high_quality_snp_positions").list.len() >= 4)
        .filter("is_high_quality_read")
        .filter(pl.col("CO_active_interval_crossover_prob") > 0)
        .collect(streaming=True)
    )

CO_source_df = pl.concat(
    joblib.Parallel(n_jobs=-1, verbose=1)(
        joblib.delayed(read_CO_source)(filename) for filename in reads_filenames
    )
)

CO_source_interval_df = (CO_source_df
    .with_columns(
        grch37_recombining_interval_start_pos = pl.col("grch37_reference_start") + pl.col("snp_positions_on_read").list.get(pl.col("idx_transitions").list.get(0)),
        grch37_recombining_interval_end_pos = pl.col("grch37_reference_start") + pl.col("snp_positions_on_read").list.get(pl.col("idx_transitions").list.get(-1) + 1),
        grch38_recombining_interval_start_pos = pl.col("grch38_reference_start") + pl.col("snp_positions_on_read").list.get(pl.col("idx_transitions").list.get(0)),
        grch38_recombining_interval_end_pos = pl.col("grch38_reference_start") + pl.col("snp_positions_on_read").list.get(pl.col("idx_transitions").list.get(-1) + 1),
    )
    .with_columns(
        grch37_recombining_interval_length = pl.col("grch37_recombining_interval_end_pos") - pl.col("grch37_recombining_interval_start_pos"),
        grch38_recombining_interval_length = pl.col("grch38_recombining_interval_end_pos") - pl.col("grch38_recombining_interval_start_pos"),
    )
)

dfs = []
for [chrom], df in CO_source_interval_df.partition_by(by=["chrom"], as_dict=True).items():
    rate_map = rate_maps[chrom]
    dfs.append(
        df.with_columns(
            grch37_recombining_interval_start_poses_cm = rate_map.get_cumulative_mass(df["grch37_recombining_interval_start_pos"]) * 1e2,
            grch37_recombining_interval_end_poses_cm = rate_map.get_cumulative_mass(df["grch37_recombining_interval_end_pos"]) * 1e2,
        ).with_columns(
            grch37_recombining_interval_cM = (pl.col("grch37_recombining_interval_end_poses_cm") - pl.col("grch37_recombining_interval_start_poses_cm")),
            grch37_cM_per_bp_across_recombining_interval = (pl.col("grch37_recombining_interval_end_poses_cm") - pl.col("grch37_recombining_interval_start_poses_cm")) / pl.col("grch37_recombining_interval_length"),
        )
    )

CO_source_interval_df = pl.concat(dfs)

co_interval_rate = (CO_source_interval_df["grch37_cM_per_bp_across_recombining_interval"] * 1e6).mean()
co_interval_rate_no_contamination = (
    CO_source_interval_df
    .filter(~pl.col("is_contamination"))
    ["grch37_cM_per_bp_across_recombining_interval"]
    .mean() * 1e6
)

{
    "CO interval cM/Mb": co_interval_rate,
    "CO interval cM/Mb no contamination": co_interval_rate_no_contamination,
}

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=-1)]: Done  18 tasks      | elapsed:    2.1s
/nfs/users/nfs_r/rs42/.local/lib/python3.11/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
[Parallel(n_jobs=-1)]: Done 168 tasks      | elapsed:    8.7s


CPU times: user 936 ms, sys: 99.5 ms, total: 1.04 s
Wall time: 12.6 s


[Parallel(n_jobs=-1)]: Done 330 out of 330 | elapsed:   12.5s finished


{'CO interval cM/Mb': 29.329306315300183,
 'CO interval cM/Mb no contamination': 30.464216731744276}

## NCO and complex subsets

In [7]:
%%time
def read_NCO_source(filename):
    return (
        pl.scan_parquet(filename)
        .select(
            'read_name',
            'chrom',
            'high_quality_snp_positions',
            'grch38_reference_start',
            'AA_motif_center_pos',
            'CO_active_interval_crossover_prob',
            'is_high_quality_read',
            'high_quality_classification_class',
            'snp_positions_on_read',
            'idx_transitions',
            'is_contamination',
        )
        .filter("is_high_quality_read")
        .filter(~pl.col("is_contamination"))
        .filter(pl.col("high_quality_classification_class") == "GC")
        .filter(pl.col("high_quality_snp_positions").list.len() >= 3)
        .filter(pl.col("CO_active_interval_crossover_prob") > 0)
        .collect(streaming=True)
    )

def read_complex_source(filename):
    return (
        pl.scan_parquet(filename)
        .select(
            'AA_motif_center_pos',
            'CO_active_interval_crossover_prob',
            'is_high_quality_read',
            'high_quality_classification_class',
            'is_contamination',
        )
        .filter("is_high_quality_read")
        .filter(~pl.col("is_contamination"))
        .filter(pl.col("high_quality_classification_class") == "CNCO")
        .filter(pl.col("CO_active_interval_crossover_prob") > 0)
        .collect(streaming=True)
    )

NCO_source_df = pl.concat(
    joblib.Parallel(n_jobs=-1, verbose=1)(
        joblib.delayed(read_NCO_source)(filename) for filename in reads_filenames
    )
)

NCO_source_df = NCO_source_df.with_columns(
    grch38_recombining_interval_start_pos = pl.col("grch38_reference_start") + pl.col("snp_positions_on_read").list.get(pl.col("idx_transitions").list.get(0)),
    grch38_recombining_interval_end_pos = pl.col("grch38_reference_start") + pl.col("snp_positions_on_read").list.get(pl.col("idx_transitions").list.get(-1) + 1),
)

complex_filtered_df = pl.concat(
    joblib.Parallel(n_jobs=-1, verbose=1)(
        joblib.delayed(read_complex_source)(filename) for filename in reads_filenames
    )
)

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=-1)]: Done  18 tasks      | elapsed:    1.5s
[Parallel(n_jobs=-1)]: Done 168 tasks      | elapsed:    7.2s
[Parallel(n_jobs=-1)]: Done 330 out of 330 | elapsed:   10.4s finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=-1)]: Done  18 tasks      | elapsed:    0.1s


CPU times: user 1.03 s, sys: 65.4 ms, total: 1.1 s
Wall time: 11.1 s


[Parallel(n_jobs=-1)]: Done 296 tasks      | elapsed:    0.7s
[Parallel(n_jobs=-1)]: Done 330 out of 330 | elapsed:    0.7s finished


## DSB-PRDM9 motifs

In [8]:
sperm_events_df = accepted_events_df.filter(pl.col("dataset").is_in(["TwinsUK", "SL"]))

motif_event_summary = (
    accepted_events_df
    .group_by("dataset", "event_type")
    .agg(
        n = pl.len(),
        DSB_PRDM9_overlap = pl.col("DSB_PRDM9_motif_strand").is_not_null().mean(),
    )
    .sort("dataset", "event_type")
)

motif_event_summary

dataset,event_type,n,DSB_PRDM9_overlap
str,str,u32,f64
"""Platinum""","""CO""",11,0.181818
"""Platinum""","""NCO""",369,0.124661
"""Platinum""","""ambiguous""",143,0.146853
"""Platinum""","""complex""",9,0.0
"""SL""","""CO""",2330,0.707725
"""SL""","""NCO""",835,0.473054
"""SL""","""ambiguous""",1846,0.584507
"""SL""","""complex""",35,0.371429
"""TwinsUK""","""CO""",4813,0.671307


In [9]:
%%time
def read_DSB_counts(filename):
    return (
        pl.scan_parquet(filename)
        .select(
            'grch38_reference_start',
            'high_quality_snp_positions',
            'CO_active_interval_crossover_prob',
            'AA_motif_center_pos',
            'is_high_quality_read',
            'high_quality_classification_class',
        )
        .filter(pl.col("high_quality_snp_positions").list.len() >= 4)
        .filter("is_high_quality_read")
        .filter(pl.col("CO_active_interval_crossover_prob") > 0)
        .filter(pl.col("grch38_reference_start").is_not_null())
        .select(
            all_n = pl.len(),
            all_has = pl.col("AA_motif_center_pos").is_not_null().sum(),
            co_n = (pl.col("high_quality_classification_class") == "CO").sum(),
            co_has = ((pl.col("high_quality_classification_class") == "CO") & pl.col("AA_motif_center_pos").is_not_null()).sum(),
        )
        .collect(streaming=True)
    )

DSB_counts_df = pl.concat(
    joblib.Parallel(n_jobs=-1, verbose=1)(
        joblib.delayed(read_DSB_counts)(filename) for filename in reads_filenames
    )
)

DSB_counts = DSB_counts_df.select(pl.all().sum()).row(0, named=True)
all_n = DSB_counts["all_n"]
all_has = DSB_counts["all_has"]
co_n = DSB_counts["co_n"]
co_has = DSB_counts["co_has"]

co_dsb_overlap = co_has / co_n
all_dsb_overlap = all_has / all_n
co_dsb_fisher = scipy.stats.fisher_exact([[co_has, co_n - co_has], [all_has, all_n - all_has]]).pvalue

nco_dsb_overlap = (NCO_source_df
    .filter(pl.col("grch38_reference_start").is_not_null())
    .select(pl.col("AA_motif_center_pos").is_not_null())
    .mean()
    .item(0,0)
)
nco_has = NCO_source_df.filter(pl.col("grch38_reference_start").is_not_null())["AA_motif_center_pos"].is_not_null().sum()
nco_n = NCO_source_df.filter(pl.col("grch38_reference_start").is_not_null()).height
nco_dsb_fisher = scipy.stats.fisher_exact([[nco_has, nco_n - nco_has], [all_has, all_n - all_has]]).pvalue

blood_nco_dsb_overlap = (accepted_events_df
    .filter(pl.col("dataset") == "Platinum")
    .filter(pl.col("event_type") == "NCO")
    ["DSB_PRDM9_motif_strand"].is_not_null().mean()
)

co_motif_inside_interval = (CO_source_interval_df
    .filter(pl.col("grch38_reference_start").is_not_null())
    .filter(pl.col("AA_motif_center_pos").is_not_null())
    .select(
        (pl.col("grch38_recombining_interval_start_pos") <= pl.col("AA_motif_center_pos")) &
        (pl.col("grch38_recombining_interval_end_pos") > pl.col("AA_motif_center_pos"))
    )
    .mean().item(0,0)
)

nco_motif_inside_interval = (NCO_source_df
    .filter(pl.col("grch38_reference_start").is_not_null())
    .filter(pl.col("AA_motif_center_pos").is_not_null())
    .select(
        (pl.col("grch38_recombining_interval_start_pos") <= pl.col("AA_motif_center_pos")) &
        (pl.col("grch38_recombining_interval_end_pos") > pl.col("AA_motif_center_pos"))
    )
    .mean().item(0,0)
)

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=-1)]: Done  18 tasks      | elapsed:    1.4s
[Parallel(n_jobs=-1)]: Done 168 tasks      | elapsed:    5.7s


CPU times: user 782 ms, sys: 55.7 ms, total: 838 ms
Wall time: 8.51 s


[Parallel(n_jobs=-1)]: Done 330 out of 330 | elapsed:    8.4s finished


In [10]:
meme_events_df = pl.read_csv(
    "/lustre/scratch122/tol/projects/sperm/analysis/prmd9_motif_analysis/recombination_events_sperm_20250417.meme_annotated.csv",
    infer_schema_length=10000,
)
sperm_meme_events_df = meme_events_df.filter(pl.col("dataset").is_in(["TwinsUK", "SL"]))

co_non_dsb_prdm9 = (sperm_meme_events_df
    .filter(pl.col("event_type") == "CO")
    .filter(pl.col("DSB_PRDM9_motif_center_pos").is_null())
    .select(pl.col("prdm9_intervals").is_not_null())
    .mean().item(0,0)
)
nco_non_dsb_prdm9 = (sperm_meme_events_df
    .filter(pl.col("event_type") == "NCO")
    .filter(pl.col("DSB_PRDM9_motif_center_pos").is_null())
    .select(pl.col("prdm9_intervals").is_not_null())
    .mean().item(0,0)
)

rr_df = (
    pl.read_csv("/lustre/scratch122/tol/projects/sperm/analysis/prmd9_motif_analysis/randomly_sampled_reads.meme_annotated.csv")
    .select("read_name", pl.col("prdm9_intervals").is_not_null().alias("has_motif"))
)

def read_random_motif_hits(filename):
    return (
        pl.scan_parquet(filename)
        .select("read_name", "AA_motif_center_pos")
        .join(rr_df.lazy(), on="read_name")
        .collect(streaming=True)
    )

random_reads_with_motifs_df = pl.concat(
    joblib.Parallel(n_jobs=-1, verbose=1)(
        joblib.delayed(read_random_motif_hits)(filename) for filename in reads_filenames
    )
)

meme_background = (random_reads_with_motifs_df
    .filter(pl.col("AA_motif_center_pos").is_null())
    .select(pl.col("has_motif").mean())
    .item(0,0)
)

complex_prdm9 = complex_filtered_df.select(pl.col("AA_motif_center_pos").is_not_null()).mean().item(0,0)
complex_prdm9_p = scipy.stats.binomtest(
    int(complex_filtered_df["AA_motif_center_pos"].is_not_null().sum()),
    complex_filtered_df.height,
    0.25,
).pvalue

motif_summary = {
    "CO DSB-PRDM9 overlap": co_dsb_overlap,
    "all reads DSB-PRDM9 overlap": all_dsb_overlap,
    "CO DSB-PRDM9 Fisher P": co_dsb_fisher,
    "CO non-DSB PRDM9 motif": co_non_dsb_prdm9,
    "all non-DSB PRDM9 motif": meme_background,
    "NCO DSB-PRDM9 overlap": nco_dsb_overlap,
    "NCO DSB-PRDM9 Fisher P": nco_dsb_fisher,
    "NCO non-DSB PRDM9 motif": nco_non_dsb_prdm9,
    "blood NCO DSB-PRDM9 overlap": blood_nco_dsb_overlap,
    "CO motif inside interval": co_motif_inside_interval,
    "NCO motif inside interval": nco_motif_inside_interval,
    "complex PRDM9 motif": complex_prdm9,
    "complex PRDM9 binomial P": complex_prdm9_p,
    "MEME background": meme_background,
}

motif_summary

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=-1)]: Done  18 tasks      | elapsed:    1.7s
[Parallel(n_jobs=-1)]: Done 168 tasks      | elapsed:    7.1s
[Parallel(n_jobs=-1)]: Done 330 out of 330 | elapsed:   11.1s finished


{'CO DSB-PRDM9 overlap': 0.6712507237984945,
 'all reads DSB-PRDM9 overlap': 0.1355114820686847,
 'CO DSB-PRDM9 Fisher P': np.float64(0.0),
 'CO non-DSB PRDM9 motif': 0.5819708351745471,
 'all non-DSB PRDM9 motif': 0.2492828415332055,
 'NCO DSB-PRDM9 overlap': 0.4545030331311246,
 'NCO DSB-PRDM9 Fisher P': np.float64(1.5964278038240434e-280),
 'NCO non-DSB PRDM9 motif': 0.4319571865443425,
 'blood NCO DSB-PRDM9 overlap': 0.12466124661246612,
 'CO motif inside interval': 0.5814011468461731,
 'NCO motif inside interval': 0.6201232032854209,
 'complex PRDM9 motif': 0.2127659574468085,
 'complex PRDM9 binomial P': np.float64(0.27337870580236745),
 'MEME background': 0.2492828415332055}

## Summary

In [11]:
key_values = {
    "CO interval cM/Mb": co_interval_rate,
    "CO interval cM/Mb no contamination": co_interval_rate_no_contamination,
    **motif_summary,
}

key_values

{'CO interval cM/Mb': 29.329306315300183,
 'CO interval cM/Mb no contamination': 30.464216731744276,
 'CO DSB-PRDM9 overlap': 0.6712507237984945,
 'all reads DSB-PRDM9 overlap': 0.1355114820686847,
 'CO DSB-PRDM9 Fisher P': np.float64(0.0),
 'CO non-DSB PRDM9 motif': 0.5819708351745471,
 'all non-DSB PRDM9 motif': 0.2492828415332055,
 'NCO DSB-PRDM9 overlap': 0.4545030331311246,
 'NCO DSB-PRDM9 Fisher P': np.float64(1.5964278038240434e-280),
 'NCO non-DSB PRDM9 motif': 0.4319571865443425,
 'blood NCO DSB-PRDM9 overlap': 0.12466124661246612,
 'CO motif inside interval': 0.5814011468461731,
 'NCO motif inside interval': 0.6201232032854209,
 'complex PRDM9 motif': 0.2127659574468085,
 'complex PRDM9 binomial P': np.float64(0.27337870580236745),
 'MEME background': 0.2492828415332055}